# FlowUp Fraud Detection - EDA & Training

This notebook provides:
1. Exploratory Data Analysis (EDA) of the credit card fraud dataset
2. Feature distribution visualization
3. Model training and comparison (XGBoost vs RandomForest)
4. Threshold optimization with Youden's J statistic
5. Performance evaluation and metrics

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.dataset_loader import load_dataset
from src.data.preprocessor import preprocess_dataframe
from src.models.trainer import run_training

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', 35)

print('Setup complete!')

## 1. Load Dataset

In [ ]:
df = load_dataset()
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["Class"].mean()*100:.4f}%')
print(f'Fraud count: {df["Class"].sum()}')
df.head()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Class'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution (Count)')
axes[0].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)

df['Class'].value_counts(normalize=True).plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Class Distribution (Proportion)')
axes[1].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)

plt.tight_layout()
plt.show()

## 3. Feature Distributions

In [ ]:
fig, axes = plt.subplots(4, 7, figsize=(20, 12))
features = [f'V{i}' for i in range(1, 29)]

for idx, feat in enumerate(features):
    ax = axes[idx // 7, idx % 7]
    df[df['Class']==0][feat].hist(ax=ax, bins=50, alpha=0.5, label='Legit', color='#2ecc71')
    df[df['Class']==1][feat].hist(ax=ax, bins=50, alpha=0.5, label='Fraud', color='#e74c3c')
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=7)

plt.legend()
plt.tight_layout()
plt.show()

## 4. Train Model

Run the full training pipeline (SMOTE + XGBoost + RF comparison).

In [ ]:
from pathlib import Path

results = run_training(output_dir=Path('../models'))

print(f"Best model: {results['best_model']}")
print(f"Optimal threshold: {results['optimal_threshold']}")
print(f"XGBoost PR-AUC: {results['metrics']['xgboost']['pr_auc']}")
print(f"RandomForest PR-AUC: {results['metrics']['random_forest']['pr_auc']}")

## 5. Results Summary

Compare the two models side by side.

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['PR-AUC', 'F1-Score', 'Recall'],
    'XGBoost': [
        results['metrics']['xgboost']['pr_auc'],
        results['metrics']['xgboost']['f1_score'],
        results['metrics']['xgboost']['recall'],
    ],
    'RandomForest': [
        results['metrics']['random_forest']['pr_auc'],
        results['metrics']['random_forest']['f1_score'],
        results['metrics']['random_forest']['recall'],
    ],
})

comparison.set_index('Metric', inplace=True)
print(comparison.to_string())

comparison.plot(kind='bar', figsize=(8, 4), rot=0, color=['#3498db', '#2ecc71'])
plt.title('Model Comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()